In [1]:
macro_data_path="../../../datasets/macro"
# read files from macro_data_path
import os
import pandas as pd
import numpy as np
# some files are csv and some are txt 
# read all files in the directory
def read_files_from_path(path):
    files = os.listdir(path)
    data = {}
    for file in files:
        print(file)
        if file.endswith(".csv"):
            data[file] = pd.read_csv(os.path.join(path, file))
        elif file.endswith(".txt"):
            data[file] = pd.read_table(os.path.join(path, file))
    return data
# print the first 5 rows of each file
data = read_files_from_path(macro_data_path)
for file in data:
    print(file)
    print(data[file].head(1))
    print("\n\n")

DKW_updates.csv
ebp_csv.csv
fci_g_public_monthly_3yr.csv
FEDS-Note-2873-cie-data.csv
FED_Note_Term_SOFR.csv
lmci_feds.csv
macro1_Monthly.txt
macro1_Quarterly.txt
DKW_updates.csv
         date  exp.real.short.rate.5  exp.inflation.5  real.term.prem.5  \
0  1983-01-03                 3.7092           4.3019            1.9021   

   inflation.risk.prem.5  tips.liq.prem.5  nominal.yield.raw.5  \
0                 0.2626              NaN              10.2174   

   nominal.yield.fitted.5  ic.raw.5  ic.fitted.5  ...  ic.fitted.10  \
0                 10.1758       NaN          NaN  ...           NaN   

   exp.real.short.rate.5f5  exp.inflation.5f5  real.term.prem.5f5  \
0                   3.3156             3.8337              2.9395   

   inflation.risk.prem.5f5  tips.liq.prem.5f5  nominal.yield.raw.5f5  \
0                 0.625199                NaN                10.6264   

   nominal.yield.fitted.5f5  ic.raw.5f5  ic.fitted.5f5  
0                 10.713999         NaN            NaN

In [2]:
# unify the column names
# change "date","DATE","period" to "date"

for file in data:
    if "date" in data[file].columns:
        data[file] = data[file].rename(columns={"date":"date"})
    elif "DATE" in data[file].columns:
        data[file] = data[file].rename(columns={"DATE":"date"})
    elif "period" in data[file].columns:
        data[file] = data[file].rename(columns={"period":"date"})
    else:
        print("no date column in {}".format(file))

import pandas as pd

# List of possible date formats
date_formats = [
    "%Y-%m-%d",    # 1999-3-31, 2000-01-01
    "%d-%m-%Y",    # 05-12-2021
    "%Ym%m",       # 1976m8
]

# Iterate through the files
for file in data:
    if "date" in data[file].columns:
        date_column = data[file]["date"]
        converted_date = None
        print(f"Converting dates in {file}")
        print("date before conversion",date_column.head(5))

        # Try each format
        for fmt in date_formats:
            try:
                converted_date = pd.to_datetime(date_column, format=fmt, errors='coerce')
                # Break if parsing succeeds (i.e., no more NaT values)
                if not converted_date.isna().all():
                    break
            except Exception as e:
                continue

        # Fallback to automatic parsing for unhandled formats
        if converted_date is None or converted_date.isna().any():
            try:
                converted_date = pd.to_datetime(date_column, errors='coerce')
            except Exception as e:
                print(f"Error parsing dates in file {file}: {e}")

        # Check for remaining NaT values and log them
        if converted_date.isna().any():
            print(f"Date format error in {file}: ", data[file].loc[converted_date.isna(), "date"].values)

        # Retain only month and year
        data[file]["date"] = converted_date.dt.to_period('M').dt.to_timestamp()
        print(f"Converted dates in {file} to {converted_date.dt.to_period('M').dt.to_timestamp().dt.strftime('%Y-%m')}")

    else:
        print(f"No date column in {file}")

    

Converting dates in DKW_updates.csv
date before conversion 0    1983-01-03
1    1983-01-04
2    1983-01-05
3    1983-01-06
4    1983-01-07
Name: date, dtype: object
Converted dates in DKW_updates.csv to 0        1983-01
1        1983-01
2        1983-01
3        1983-01
4        1983-01
          ...   
11255    2026-02
11256    2026-02
11257    2026-02
11258    2026-02
11259    2026-02
Name: date, Length: 11260, dtype: object
Converting dates in ebp_csv.csv
date before conversion 0    1973-01-01
1    1973-02-01
2    1973-03-01
3    1973-04-01
4    1973-05-01
Name: date, dtype: object
Converted dates in ebp_csv.csv to 0      1973-01
1      1973-02
2      1973-03
3      1973-04
4      1973-05
        ...   
633    2025-10
634    2025-11
635    2025-12
636    2026-01
637    2026-02
Name: date, Length: 638, dtype: object
Converting dates in fci_g_public_monthly_3yr.csv
date before conversion 0    1990-01-01
1    1990-02-01
2    1990-03-01
3    1990-04-01
4    1990-05-01
Name: date, dtype:

In [3]:
# cast all non-date columns to numeric
for file in data:
    for column in data[file].columns:
        if column != "date":
            data[file][column] = pd.to_numeric(data[file][column], errors='coerce')

In [4]:
# group mean by date
for file in data:
    data[file] = data[file].groupby("date").mean()
    # print(data[file].head(5))
    # still use date as a column
    data[file].reset_index(inplace=True)

In [5]:
# merge all data by date column
merged_data = None
for file in data:
    if merged_data is None:
        merged_data = data[file]
    else:
        merged_data = pd.merge(merged_data, data[file], on="date", how="outer")

In [6]:
# select date range from 2000-01 to 2024-12
merged_data = merged_data[(merged_data["date"] >= "2000-01") & (merged_data["date"] <= "2024-12")]

In [7]:
# describe the data, check nan values of each column
# forward fill nan values, then backfill for any leading NaNs
merged_data = merged_data.sort_values("date")
merged_data = merged_data.ffill()
merged_data = merged_data.bfill()
print(merged_data.describe())

                      date  exp.real.short.rate.5  exp.inflation.5  \
count                  300             300.000000       300.000000   
mean   2012-06-16 02:14:24               0.076344         2.399521   
min    2000-01-01 00:00:00              -0.987738         1.664300   
25%    2006-03-24 06:00:00              -0.730200         2.102921   
50%    2012-06-16 00:00:00              -0.171234         2.427825   
75%    2018-09-08 12:00:00               0.884330         2.664486   
max    2024-12-01 00:00:00               2.340100         3.309330   
std                    NaN               0.924723         0.362170   

       real.term.prem.5  inflation.risk.prem.5  tips.liq.prem.5  \
count        300.000000             300.000000       300.000000   
mean           0.238235               0.029346         0.545336   
min           -0.381491              -0.088627        -1.051996   
25%            0.030236              -0.012774         0.252576   
50%            0.193238           

In [8]:
# save the merged data to a csv file
data_folder = "../../../datasets/macro_processed"
# change the 'date' column to "Date"
merged_data = merged_data.rename(columns={"date":"Date"})
# create the folder if it does not exist
if not os.path.exists(data_folder):
    os.makedirs(data_folder)
merged_data.to_csv(os.path.join(data_folder, "macro_data.csv"), index=False)

In [11]:
#read all stock data from the folder
import os
import pandas as pd
stock_data_path = "../../../workdir/yahoofinance_day_prices_dj30"
# read all csv files in the directory
def read_files_from_path(path):
    files = os.listdir(path)
    data = {}
    for file in files:
        if file.endswith(".csv"):
            filepath = os.path.join(path, file)
            if not os.path.isfile(filepath):
                continue
            file_name = file.split(".")[0]
            try:
                df = pd.read_csv(filepath)
            except pd.errors.EmptyDataError:
                print(f"Skipping empty file: {file}")
                continue
            if len(df) == 0:
                print(f"Skipping file with no rows: {file}")
                continue
            data[file_name] = df
    return data
# print the first 5 rows of each file
data = read_files_from_path(stock_data_path)
print(f"Loaded {len(data)} tickers: {sorted(data.keys())}")
# concatenate all stock data together into one dataframe, and add a column "ticker" to indicate the stock
stock_data = None
for file in data:
    ticker = file.split(".")[0]
    if stock_data is None:
        stock_data = data[file]
        stock_data["ticker"] = ticker
    else:
        # concatenate the stock data raw by raw
        stock_data_ = data[file]
        stock_data_["ticker"] = ticker
        stock_data = pd.concat([stock_data, data[file]], axis=0)
unique_dates=stock_data["Date"].unique()
print("unique dates",len(unique_dates))
# drop the row where Date is NaN
stock_data = stock_data.dropna(subset=["Date"])
print(stock_data.columns)
print(stock_data.head(5))
print("date range of stock data",stock_data["Date"].min(),stock_data["Date"].max())
print("size of stock data",stock_data.shape)
print("unique tickers",stock_data["ticker"].unique())
# print the NaN numb of each column
print(stock_data.isna().sum())
# save the merge stock data to a csv file
# sort the stock data by date
stock_data = stock_data.sort_values("Date")
data_folder = "../../../datasets/stock_data"
# create the folder if it does not exist
if not os.path.exists(data_folder):
    os.makedirs(data_folder)
stock_data.to_csv(os.path.join(data_folder, "megred_stock_data.csv"), index=False)

Skipping empty file: WBA.csv
Loaded 28 tickers: ['AAPL', 'AMGN', 'AXP', 'BA', 'CAT', 'CRM', 'CSCO', 'CVX', 'DIS', 'GS', 'HD', 'HON', 'IBM', 'INTC', 'JNJ', 'JPM', 'KO', 'MCD', 'MMM', 'MRK', 'MSFT', 'NKE', 'PG', 'TRV', 'UNH', 'V', 'VZ', 'WMT']
unique dates 6037
Index(['Date', 'Adj Close', 'Close', 'High', 'Low', 'Open', 'Volume',
       'ticker'],
      dtype='object')
         Date  Adj Close     Close      High       Low      Open     Volume  \
0  2000-01-03   0.838496  0.999442  1.004464  0.907924  0.936384  535796800   
1  2000-01-04   0.767802  0.915179  0.987723  0.903460  0.966518  512377600   
2  2000-01-05   0.779038  0.928571  0.987165  0.919643  0.926339  778321600   
3  2000-01-06   0.711621  0.848214  0.955357  0.848214  0.947545  767972800   
4  2000-01-07   0.745330  0.888393  0.901786  0.852679  0.861607  460734400   

  ticker  
0   AAPL  
1   AAPL  
2   AAPL  
3   AAPL  
4   AAPL  
date range of stock data 2000-01-03 2023-12-29
size of stock data (165851, 8)
unique tick

In [12]:
import pandas as pd
# merge the stock data with the macro data
stock_data_path="../../../datasets/stock_data/megred_stock_data.csv"
selected_macro_path = "macro_list.txt"
with open(selected_macro_path, "r") as f:
    Macro_features = f.read().splitlines()
macro_data_path="../../../datasets/macro_processed/macro_data.csv"
# merge the stock data with the macro data
stock_data = pd.read_csv(stock_data_path)
full_macro_data = pd.read_csv(macro_data_path)
selected_macro_data = full_macro_data[['Date']+Macro_features]
print("date stamps of macro data",selected_macro_data.head(10))
# now selected_macro_data only have the first day of each month, we need to forward fill the data to all days in the month
selected_macro_data["Date"] = pd.to_datetime(selected_macro_data["Date"])
selected_macro_data.set_index("Date", inplace=True)
selected_macro_data = selected_macro_data.resample("D").ffill()
print("resampled macro data",selected_macro_data.head(10))
# save the resampled macro data to a csv file
data_folder = "../../../datasets/macro_processed"
# create the folder if it does not exist
if not os.path.exists(data_folder):
    os.makedirs(data_folder)
selected_macro_data.to_csv(os.path.join(data_folder, "macro_data_resampled.csv"))

date stamps of macro data          Date  exp.real.short.rate.5  exp.inflation.5  real.term.prem.5  \
0  2000-01-01               2.116505         3.309330          0.982340   
1  2000-02-01               2.224655         3.272955          0.955350   
2  2000-03-01               2.215509         3.216578          0.863452   
3  2000-04-01               2.147000         3.150437          0.779137   
4  2000-05-01               2.340100         3.256609          0.886268   
5  2000-06-01               2.213023         3.166773          0.779450   
6  2000-07-01               2.188380         3.140695          0.729120   
7  2000-08-01               2.169926         3.091196          0.650504   
8  2000-09-01               2.095315         3.090670          0.630465   
9  2000-10-01               2.064981         3.060181          0.562281   

   inflation.risk.prem.5  tips.liq.prem.5  nominal.yield.raw.5  \
0               0.135650         1.295855             6.524823   
1               

C:\Users\jloya\AppData\Local\Temp\ipykernel_22492\1508866237.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  selected_macro_data["Date"] = pd.to_datetime(selected_macro_data["Date"])
